# Laplacian Identity Recovery

Viewpoint lives in the *lowest* Laplacian eigenvectors (the slowest variation on
the manifold). Identity is a much finer structure — same individual = near-duplicate
Fisher vectors. This notebook tests whether the graph Laplacian captures identity
at all, and if so, where.

## Hypotheses
1. **EVs 1-3 (viewpoint subspace) fail at reID** — they encode the slow "which side of the zebra" variation, not per-individual detail.
2. **Higher EVs might help** — once viewpoint is factored out, finer structure should live in the middle-to-high EVs.
3. **Per-viewpoint subgraphs are the right place** — restrict to a single viewpoint and the dominant variation *is* identity. EVs 1-3 of the per-viewpoint graph should encode individuals.
4. **Raw FV cosine is the baseline to beat** — if the Laplacian adds no new information for identity, we should fall back to it.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import pickle
import time
from pathlib import Path
from collections import Counter, defaultdict

from src.config.config import MainConfig
from src.data.preprocessed_dataset import PreprocessedDataset
from src.data.coco_loader import COCOLoader
from src.evaluation import (
    load_or_compute_matching, get_identity_mapping, compute_reid_accuracy,
    COARSE_MAP, coarse_viewpoint, fisher_discriminant_1d,
)
from src.codebook.gmm_trainer import load_gmm_model
from src.features.fisher_vector import build_block_mask, normalize_fvs
from src.laplacian import build_knn_graph, normalized_laplacian, smallest_eigenvectors

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'


In [2]:
# ============================================================
# Load config, dataset, raw FVs, matching, identity
# ============================================================
config = MainConfig.from_yaml(Path('config_zebra_test.yaml'))
dataset = PreprocessedDataset(config.output_root)
coco_loader = COCOLoader(config.coco_json_path, config.dataset_root)

# Raw weight Fisher vectors
raw_pkl_path = config.output_root / 'weight_fisher_vectors_raw.pkl'
print(f'Loading raw weight FVs from {raw_pkl_path}')
with open(raw_pkl_path, 'rb') as f:
    raw_data = pickle.load(f)
all_det_ids = raw_data['det_ids']
all_fvs_raw = raw_data['fvs_raw']
del raw_data

gmm, _ = load_gmm_model(config.gmm_model_path)
K = gmm.n_components
D = gmm.means_.shape[1]
block = 2 * D + 1
print(f'GMM: K={K}, D={D}, FV dim={K * block}')
print(f'Raw FVs: {len(all_det_ids)} detections, shape={all_fvs_raw.shape}')

# GT matching
matched = load_or_compute_matching(
    dataset, coco_loader, config.output_root,
    target_size=config.active_resize_size,
    patch_size=config.active_patch_size,
    category_names=config.matching_categories,
    min_overlap_fraction=0.5,
)
print(f'Matched: {len(matched)} detections')

# Identity + image lookup
identity_map = get_identity_mapping(matched)   # det_id -> individual_id
det_to_image = {m.detection_id: m.gt_annotation.image_uuid for m in matched}
det_to_viewpoint = {m.detection_id: m.gt_annotation.viewpoint for m in matched}

print(f'Identity labels: {len(identity_map)}')
print(f'Unique individuals: {len(set(identity_map.values()))}')


Loading raw weight FVs from /fs/ess/PAS2136/ggr_data/dino_data_2811/weight_fisher_vectors_raw.pkl
GMM: K=64, D=256, FV dim=32832
Raw FVs: 8271 detections, shape=(8271, 32832)
Loaded 7486 matching results from cache (/fs/ess/PAS2136/ggr_data/dino_data_2811/matching/gt_matching.json)
Matched: 7486 detections
Identity labels: 7486
Unique individuals: 596


In [ ]:
# ============================================================
# Settings + filter to reID-valid subset
# ============================================================
USE_WEIGHT = True
USE_MEAN   = True
USE_VAR    = True
KNN_K      = 10
RANDOM_SEED = 42

# Build FV mask + normalize
fv_mask = build_block_mask(K, D, USE_WEIGHT, USE_MEAN, USE_VAR)
all_fvs_norm = normalize_fvs(all_fvs_raw, fv_mask)
print(f'FV mask active dims: {fv_mask.sum()}/{len(fv_mask)}')

# --- Filter to detections that have identity + non-singleton ---
ind_counts = Counter(identity_map.values())
valid_inds = {ind for ind, c in ind_counts.items() if c >= 2}
print(f'Individuals with >=2 detections: {len(valid_inds)} / {len(ind_counts)}')

det_id_to_idx = {d: i for i, d in enumerate(all_det_ids)}

valid_dets = []
valid_vps = []
for det_id in all_det_ids:
    if det_id not in identity_map:
        continue
    if identity_map[det_id] not in valid_inds:
        continue
    vp_raw = det_to_viewpoint.get(det_id)
    if vp_raw in (None, 'ignore', 'unknown'):
        continue
    coarse = coarse_viewpoint(vp_raw)
    if coarse is None:
        continue
    valid_dets.append(det_id)
    valid_vps.append(coarse)

valid_vps = np.array(valid_vps)
valid_fv_idx = np.array([det_id_to_idx[d] for d in valid_dets])
valid_fvs = all_fvs_norm[valid_fv_idx]
print(f'\nValid reID detections: {len(valid_dets)}')
print(f'Viewpoint breakdown:')
for vp, c in sorted(Counter(valid_vps).items(), key=lambda kv: -kv[1]):
    print(f'  {vp:>8}: {c:5d}')

# Subset identity / image maps to valid dets (used by compute_reid_accuracy)
valid_identity_map = {d: identity_map[d] for d in valid_dets}
valid_det_to_image = {d: det_to_image[d] for d in valid_dets}

del all_fvs_norm  # free memory


In [ ]:
# ============================================================
# Helpers: wrap compute_reid_accuracy for quick embedding evaluation
# ============================================================

def eval_reid_embedding(embedding, label, det_ids=None, id_map=None, img_map=None, verbose=True):
    """Evaluate reID on a 2D embedding [N, d] with rows aligned to det_ids.
    compute_reid_accuracy L2-normalizes internally and uses cosine similarity."""
    if det_ids is None:
        det_ids = valid_dets
    if id_map is None:
        id_map = valid_identity_map
    if img_map is None:
        img_map = valid_det_to_image
    fv_dict = {det_ids[i]: embedding[i] for i in range(len(det_ids))}
    t0 = time.time()
    metrics = compute_reid_accuracy(
        fisher_vectors=fv_dict,
        identity_mapping=id_map,
        exclude_same_image=True,
        detection_to_image=img_map,
    )
    elapsed = time.time() - t0
    if verbose:
        print(f'{label:>35}  top1={metrics.top1_accuracy:6.2%}  '
              f'top5={metrics.top5_accuracy:6.2%}  '
              f'top10={metrics.top10_accuracy:6.2%}  '
              f'mrr={metrics.mean_reciprocal_rank:.4f}  '
              f'n={metrics.num_queries}  ({elapsed:.1f}s)')
    return metrics


## Section 1 — Baselines

1. Raw FV cosine (gold standard for identity in this project).
2. Laplacian 3D (EVs 1-3) — viewpoint subspace. Expected to be *bad* for identity.


In [5]:
print(f'{"Method":>35}  {"top1":>6}  {"top5":>6}  {"top10":>6}  {"mrr":>6}   n    time')
print('-' * 100)
metrics_fv = eval_reid_embedding(valid_fvs, 'Raw FV cosine (baseline)')


                             Method    top1    top5   top10     mrr   n    time
----------------------------------------------------------------------------------------------------
           Raw FV cosine (baseline)  top1=66.89%  top5=77.65%  top10=81.16%  mrr=0.7198  n=7230  (10.1s)


## Section 2 — Full-graph Laplacian

Build the kNN graph on all valid FVs, compute the 50 smallest Laplacian eigenvectors,
then sweep reID accuracy over various EV subsets. If identity lives anywhere in the
Laplacian spectrum, one of these subsets should win.


In [ ]:
# Build full-graph Laplacian and compute 50 smallest EVs
print(f'Building {KNN_K}-NN graph on {len(valid_fvs)} FVs...')
t0 = time.time()
A_full, deg_full = build_knn_graph(valid_fvs, k=KNN_K)
L_full = normalized_laplacian(A_full, deg_full)
print(f'  Built in {time.time()-t0:.1f}s')

N_EIG = 50
eigvals_full, eigvecs_full = smallest_eigenvectors(L_full, N_EIG)
print(f'Eigenvector matrix: {eigvecs_full.shape}')


In [7]:
# ============================================================
# reID sweep over EV subsets
# ============================================================
# Drop EV0 (the constant), then try various slices.
print(f'{"Method":>35}  {"top1":>6}  {"top5":>6}  {"top10":>6}  {"mrr":>6}   n    time')
print('-' * 100)

# Baseline reference
eval_reid_embedding(valid_fvs, 'Raw FV cosine (reference)')

# EV subsets to test
ev_subsets = [
    ('Laplacian EVs 1-3 (viewpoint)', slice(1, 4)),
    ('Laplacian EVs 1-10',            slice(1, 11)),
    ('Laplacian EVs 1-30',            slice(1, 31)),
    ('Laplacian EVs 1-50',            slice(1, 51)),
    ('Laplacian EVs 4-10 (skip VP)',  slice(4, 11)),
    ('Laplacian EVs 4-30 (skip VP)',  slice(4, 31)),
    ('Laplacian EVs 4-50 (skip VP)',  slice(4, 51)),
    ('Laplacian EVs 10-50',           slice(10, 51)),
    ('Laplacian EVs 30-50',           slice(30, 51)),
]
reid_sweep = {}
for label, sl in ev_subsets:
    emb = eigvecs_full[:, sl]
    reid_sweep[label] = eval_reid_embedding(emb, label)

# FV residual: FV minus projection onto EVs 1-3 (what FV space looks
# like after removing the viewpoint subspace)
print()
ev13 = eigvecs_full[:, 1:4]    # [N, 3]
# Project each FV column onto EVs 1-3 and subtract
proj = ev13 @ (ev13.T @ valid_fvs)
fv_residual = valid_fvs - proj
reid_sweep['FV residual (minus EV 1-3)'] = eval_reid_embedding(
    fv_residual, 'FV residual (minus EV 1-3)'
)


                             Method    top1    top5   top10     mrr   n    time
----------------------------------------------------------------------------------------------------
          Raw FV cosine (reference)  top1=66.89%  top5=77.65%  top10=81.16%  mrr=0.7198  n=7230  (10.1s)
      Laplacian EVs 1-3 (viewpoint)  top1=30.08%  top5=38.35%  top10=42.12%  mrr=0.3462  n=7230  (6.4s)
                 Laplacian EVs 1-10  top1=37.47%  top5=48.45%  top10=53.07%  mrr=0.4306  n=7230  (6.3s)
                 Laplacian EVs 1-30  top1=43.24%  top5=54.50%  top10=59.42%  mrr=0.4894  n=7230  (6.3s)
                 Laplacian EVs 1-50  top1=44.92%  top5=57.98%  top10=62.75%  mrr=0.5123  n=7230  (6.2s)
       Laplacian EVs 4-10 (skip VP)  top1=36.07%  top5=47.44%  top10=51.69%  mrr=0.4179  n=7230  (6.3s)
       Laplacian EVs 4-30 (skip VP)  top1=43.02%  top5=54.59%  top10=59.45%  mrr=0.4879  n=7230  (6.3s)
       Laplacian EVs 4-50 (skip VP)  top1=45.03%  top5=57.93%  top10=62.72%  mrr=0.5127  n

## Section 3 — Per-viewpoint subgraph Laplacian

Restrict the graph to a single coarse viewpoint. Within "right" zebras, viewpoint
variation is nearly gone, so the dominant manifold structure should be **identity/pose**.
If the Laplacian can capture identity at all, this is where it should show up:
EVs 1-3 of the per-viewpoint graph should be a much better reID embedding than
EVs 1-3 of the full-graph Laplacian.


In [ ]:
# ============================================================
# For each coarse VP: build subgraph, compute 20 EVs, evaluate reID
# ============================================================
per_vp_results = {}

print(f'{"Method":>40}  {"top1":>6}  {"top5":>6}  {"top10":>6}  {"mrr":>6}   n    time')
print('-' * 105)

for vp in ['right', 'left', 'front', 'back']:
    m = valid_vps == vp
    n_vp = m.sum()
    if n_vp < 30:
        print(f'\n--- {vp}: only {n_vp} detections, skipping ---')
        continue

    print(f'\n--- {vp}: {n_vp} detections ---')
    fvs_vp  = valid_fvs[m]
    dets_vp = [valid_dets[i] for i in np.where(m)[0]]
    id_map_vp  = {d: valid_identity_map[d] for d in dets_vp}
    img_map_vp = {d: valid_det_to_image[d]  for d in dets_vp}

    # Require at least 2 detections per identity within this VP
    ind_counts_vp = Counter(id_map_vp.values())
    keep = np.array([ind_counts_vp[id_map_vp[d]] >= 2 for d in dets_vp])
    if keep.sum() < 30:
        print(f'  only {keep.sum()} non-singleton detections in {vp}, skipping')
        continue

    fvs_vp  = fvs_vp[keep]
    dets_vp = [d for d, k in zip(dets_vp, keep) if k]
    id_map_vp  = {d: id_map_vp[d]  for d in dets_vp}
    img_map_vp = {d: img_map_vp[d] for d in dets_vp}
    print(f'  after non-singleton filter: {len(dets_vp)} dets, '
          f'{len(set(id_map_vp.values()))} individuals')

    # Baseline: raw FV cosine within this VP
    per_vp_results[(vp, 'raw_fv')] = eval_reid_embedding(
        fvs_vp, f'{vp}: Raw FV cosine',
        det_ids=dets_vp, id_map=id_map_vp, img_map=img_map_vp,
    )

    # Build per-VP Laplacian
    k_vp = min(KNN_K, len(fvs_vp) - 1)
    A_vp, deg_vp = build_knn_graph(fvs_vp, k=k_vp)
    L_vp = normalized_laplacian(A_vp, deg_vp)

    n_eig_vp = min(20, len(fvs_vp) - 2)
    eigvals_vp, eigvecs_vp = smallest_eigenvectors(L_vp, n_eig_vp)

    for label_suffix, sl in [
        ('Lap EVs 1-3',  slice(1, 4)),
        ('Lap EVs 1-10', slice(1, min(11, n_eig_vp))),
        ('Lap EVs 1-20', slice(1, n_eig_vp)),
    ]:
        emb = eigvecs_vp[:, sl]
        if emb.shape[1] == 0:
            continue
        per_vp_results[(vp, label_suffix)] = eval_reid_embedding(
            emb, f'{vp}: {label_suffix}',
            det_ids=dets_vp, id_map=id_map_vp, img_map=img_map_vp,
        )


## Section 4 — Which eigenvectors actually separate identities?

For each individual EV of the full-graph Laplacian, compute a 1D Fisher discriminant
ratio: **between-identity variance / within-identity variance**. EVs with a high
ratio encode identity-aligned structure; low-ratio EVs are uninformative for reID.

This tells us whether identity signal is concentrated in a few specific EVs or
smeared across many of them.


In [ ]:
# ============================================================
# Per-EV Fisher discriminant for identity
# ============================================================
id_array = np.array([valid_identity_map[d] for d in valid_dets])
unique_inds = np.unique(id_array)
ind_to_int = {ind: i for i, ind in enumerate(unique_inds)}
id_int = np.array([ind_to_int[i] for i in id_array])

print(f'{"EV":>4} {"eigenvalue":>12} {"Fisher ratio (identity)":>25}')
print('-' * 45)
fisher_ratios = []
for i in range(N_EIG):
    ratio = fisher_discriminant_1d(eigvecs_full[:, i], id_int)
    fisher_ratios.append(ratio)
    marker = '  <-- high' if ratio > 0.5 else ''
    print(f'{i:>4} {eigvals_full[i]:>12.5f} {ratio:>25.4f}{marker}')

fisher_ratios = np.array(fisher_ratios)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(N_EIG), fisher_ratios, 'o-', color='steelblue')
ax.set_xlabel('Eigenvector index')
ax.set_ylabel('Fisher ratio (between / within identity)')
ax.set_title('Per-EV identity discriminability')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.scatter(eigvals_full[:N_EIG], fisher_ratios, c=range(N_EIG), cmap='viridis', s=50)
ax.set_xlabel('Eigenvalue (graph frequency)')
ax.set_ylabel('Fisher ratio (identity)')
ax.set_title('Identity signal vs graph frequency')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

top_evs = np.argsort(-fisher_ratios)[:10]
print(f'\nTop 10 EVs by Fisher ratio: {top_evs.tolist()}')
print(f'Their ratios: {[round(fisher_ratios[i], 4) for i in top_evs]}')


## Findings

Fill in after running the notebook. Key questions to answer:
1. Does raw FV cosine already win on reID? (Likely yes.)
2. Is there *any* Laplacian EV subset that matches or beats it?
3. Do per-viewpoint subgraph EVs do better than full-graph EVs for reID?
4. Is identity signal concentrated in a few EVs or spread across many?
